*1)Problem statement*

- When building an LLM app (chatbot / Q&A / RAG) you usually need:
- A model (LLM) to generate answers (here: Groq-hosted Llama).
- A prompt template to structure the conversation.
- Conversation memory so the bot remembers earlier messages per user/session.
- History control (trimming) so you don’t exceed context length and cost.
- Retrieval (RAG) so the model can answer using external documents instead of guessing.
- An API layer (FastAPI / LangServe) to expose the chain as an endpoint.
- This notebook code demonstrates all of the above as building blocks.

*2) Tech stack used*
- Python
- FastAPI → Web API framework (you imported it; typical usage is to host routes)
- LangChain Core → prompts, runnables, output parsers, message types
- LangServe (add_routes) → expose LangChain “chains” as API endpoints
- dotenv (load_dotenv) → load secrets from .env
- Groq + LangChain Groq (ChatGroq) → LLM backend
- Vector DB + Retriever (vectordb.as_retriever) → retrieval for RAG
- Chat message history (ChatMessageHistory) → per-session memory store

*3) High-level solution architecture*
*A) Chat (Memory + Prompt + LLM)*
- Store chat messages per session_id
- Build a prompt that includes system instruction + message history placeholder
- Send messages to the LLM
- Optionally trim message history to last N tokens

*B) RAG (Retriever + Prompt + LLM)*
- User question comes in
- Retriever fetches top-k relevant docs
- Prompt injects {context} + {question}
- LLM answers based only on retrieved context

In [ ]:
import os
from fastapi import FastAPI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langserve import add_routes
from dotenv import load_dotenv

# Load environment variables from .env file
# (Example: GROQ_API_KEY=xxxx)
load_dotenv()

from langchain_groq import ChatGroq

# Read Groq API key securely from environment variables
groq_api_key = os.getenv("GROQ_API_KEY")

# Create the LLM object (this is the "brain")
# model="llama-3.1-8b-instant" is the Groq-hosted Llama model name
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)

### *Using -Message history functionality*

In [ ]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hello this is kamal")]) 

In [ ]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hello this is kamal"),
        AIMessage(content="Hello Kamal, how are you today? Is there something I can help you with or would you like to chat?"),
        HumanMessage(content="What is my name")
    ]
)

We are trying to implement the above human , Ai message context memory chart using the library. 
ChatMessageHistory - stores the messages like user , assitant format



In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory 
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory
    if session_id not in store: 
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [ ]:
# First call
config1={"configurable":{"session_id":"chat1"}}
with_message_history.invoke(
    [HumanMessage(content="Hello this is kamal")],
    config=config1
)

In [ ]:
# Second call (same session_id = memory works)
with_message_history.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user1"}}
)

### *Prompt Template* 


In [32]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
      [
        ("system",
         "you are a helpful assitant.Answer all questions to the best of your ability in {language}."
        ),
        MessagesPlaceholder(variable_name="input"),
      ]

)

chain=prompt|model

In [34]:
response=chain.invoke({"input":[HumanMessage(content="Hi My name is kamal")],"language":"Tamil"}) 
response.content

'வணக்கம். நான் உங்களுக்கு உதவி செய்ய இருப்பேன். உங்கள் பெயர் கமல் என்பது போல் சொன்னால் சரியாக இருக்குமா?'

In [35]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="input")

In [36]:
config1={"configurable":{"session_id":"chat3"}}

In [37]:
response = with_message_history.invoke(
    {"input": [HumanMessage(content="Hi, I am Kamal")], "language": "Tamil"},
    config=config1
)
response.content

'வணக்கம் கமல்! நான் உங்களுக்கு உதவி செய்ய இங்கு இருக்கிறேன். என்ன பொருளில் உங்களுக்கு உதவி தேவைப்படுகிறது?'

### Managing the conversion history

 Trims the conversation to the most recent messages within a token limit, while preserving system instructions and complete human–AI turns.

In [53]:
from langchain_core.messages import SystemMessage,trim_messages
# Limits chat history to the last 70 model-counted tokens, keeps system messages, 
# avoids cutting messages mid-way, and ensures the trimmed history starts with a human message.
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="You're a good assistant."),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="Hi Bob! Nice to meet you. How can I help today?"),
    HumanMessage(content="can you remind me what my name is?"),
    AIMessage(content="Your name is Bob."),
    HumanMessage(content="great. suggest 3 healthy snack ideas"),
    AIMessage(content="Sure! 1) Greek yogurt + berries 2) Roasted chickpeas 3) Apple slices with peanut butter.")
]

trimmer.invoke(messages)

[SystemMessage(content="You're a good assistant.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='can you remind me what my name is?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your name is Bob.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='great. suggest 3 healthy snack ideas', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sure! 1) Greek yogurt + berries 2) Roasted chickpeas 3) Apple slices with peanut butter.', additional_kwargs={}, response_metadata={})]

itemgetter("message")
→ Takes the "message" field from the input data.

| trimmer
→ Trims the chat messages to fit within the token limit (keeps recent, valid messages).

RunnablePassthrough.assign(messages=...)
→ Adds the trimmed messages back into the input under the key "messages".

| prompt
→ Inserts those messages into the prompt template.

| model
→ Sends the prompt to the language model to generate a response.

In [61]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.messages import HumanMessage

chain = (
    RunnablePassthrough.assign(
        messages=itemgetter("input") | trimmer
    )
    | prompt
    | model
)
print (chain)

first=RunnableAssign(mapper={
  messages: RunnableLambda(itemgetter('input'))
            | RunnableLambda(...)
}) middle=[ChatPromptTemplate(input_variables=['input', 'language'], input_types={'input': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing

In [ ]:
response = chain.invoke(
    {
        "input":messages + [HumanMessage(content="how many snacks was suggested")],
        "language": "English"
    }
)

response.content

# Note : If your prompt has MessagesPlaceholder(variable_name="X"), 
# then .invoke() must receive "X": [HumanMessage(...), ...] — and your chain must not rename it to something else.

'3 snack ideas were suggested.'

In [ ]:
# With memory history 

with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="input") 
config={"configurable":{"session_id":"chat5" }} 


In [64]:
with_message_history.invoke(
    {"input": messages + [HumanMessage(content="What is my name?")],
    "language":"English",
    },
    config={"configurable": {"session_id": "user1"}}
)

AIMessage(content='Your name is Bob.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 234, 'total_tokens': 240, 'completion_time': 0.003716093, 'completion_tokens_details': None, 'prompt_time': 0.015321729, 'prompt_tokens_details': None, 'queue_time': 0.088150097, 'total_time': 0.019037822}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b42ea-027a-7e00-9d67-691b4b5980a0-0', usage_metadata={'input_tokens': 234, 'output_tokens': 6, 'total_tokens': 240})

In [67]:
import os 
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document 
from langchain_groq import ChatGroq


documents = [
    Document(
        page_content="LangChain is a framework for building applications using large language models.",
        metadata={"source": "langchain_docs"}
    ),
    Document(
        page_content="Chroma is a vector database used to store embeddings and perform similarity search.",
        metadata={"source": "vector_db_docs"}
    ),
    Document(
        page_content="Vector databases enable semantic search by comparing embeddings instead of keywords.",
        metadata={"source": "ml_basics"}
    )
]
load_dotenv() 
groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")
model=ChatGroq(groq_api_key=groq_api_key,model='llama-3.1-8b-instant')
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectordb=Chroma.from_documents(documents,embedding=embeddings) 



In [75]:
vectordb.similarity_search("cat")


[Document(id='a89a2aa0-d8d5-4555-8def-d3e3692c40ff', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications using large language models.'),
 Document(id='880af5f5-cd12-4b48-8e05-26c7274cd636', metadata={'source': 'ml_basics'}, page_content='Vector databases enable semantic search by comparing embeddings instead of keywords.'),
 Document(id='6107b524-1177-48c6-9cde-846311525bd0', metadata={'source': 'vector_db_docs'}, page_content='Chroma is a vector database used to store embeddings and perform similarity search.')]

In [ ]:
#Async query 
# Runs asynchronously (non-blocking)
# Allows other tasks to run while searching
# Must be used inside an async function
await vectordb.asimilarity_search("cat")

[Document(id='a89a2aa0-d8d5-4555-8def-d3e3692c40ff', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications using large language models.'),
 Document(id='880af5f5-cd12-4b48-8e05-26c7274cd636', metadata={'source': 'ml_basics'}, page_content='Vector databases enable semantic search by comparing embeddings instead of keywords.'),
 Document(id='6107b524-1177-48c6-9cde-846311525bd0', metadata={'source': 'vector_db_docs'}, page_content='Chroma is a vector database used to store embeddings and perform similarity search.')]

In [82]:
# Retriever ( example 1 )

from typing import List 
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda 

retriever=RunnableLambda(vectordb.similarity_search).bind(k=1)
retriever.batch(["cat","dog"])


[[Document(id='a89a2aa0-d8d5-4555-8def-d3e3692c40ff', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications using large language models.')],
 [Document(id='a89a2aa0-d8d5-4555-8def-d3e3692c40ff', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications using large language models.')]]

In [ ]:
# Retriever 2 

retriever=vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

# RunnableLambda wraps a specific function (like asimilarity_search) into a Runnable, 
# While as_retriever() returns a full-featured Retriever object with a standard interface and 
# retrieval options designed for RAG pipelines.
# In Chroma, k controls how many documents are retrieved, 
# MMR improves result diversity, and filters restrict searches using metadata before similarity matching.

[[Document(id='a89a2aa0-d8d5-4555-8def-d3e3692c40ff', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications using large language models.')],
 [Document(id='a89a2aa0-d8d5-4555-8def-d3e3692c40ff', metadata={'source': 'langchain_docs'}, page_content='LangChain is a framework for building applications using large language models.')]]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the provided context."),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

rag_chain = (
    {
        "context": retriever, 
        "question": RunnablePassthrough()
    }
    | prompt
    | model
)

response = rag_chain.invoke("tell me about dog")
print(response.content)

There is no information about dogs in the provided context. The context appears to be about a framework for building applications using large language models called LangChain.
